# 6. Fusion, and why routing does not work

Reproduces **Section 4.5**:

| output | paper |
|---|---|
| each component alone and its error correlation with the volumetric features | `tab:errcorr` |
| grading AUC as each embedding is appended, with a paired bootstrap | `tab:matrix` |
| the same fusion rows over five fold assignments | Section 4.5, the five-seed mean |
| the per-case routing alternative | Section 4.5, the negative result |

Runs from `data_cache/` alone, in a few minutes.

## The two ways of combining

**Concatenation.** The reduced embedding is appended to the seventeen volumetric features to form
one vector per case, which the classifier then grades. No per-case weighting, no selection.

**Routing.** A share of the cases is handed to a classifier over MedGemma's hidden state and the
rest stay with the volumetric classifier. MedGemma's own softmax over the grade token it wrote is
the gate: the cases it is most confident on are the ones handed over.

The paper reports concatenation and reports that routing failed. This notebook shows both, and
shows the diagnostic that explains why the failure is structural rather than a matter of tuning
the gate.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src import config
print(config.describe())

In [ ]:
from src import cohort, routing, stats, tables

blocks, y, cases = cohort.load_blocks()
cohort.inject_encoder_blocks(blocks, cases, keys=("medgemma_fewshot", "biomedclip_enh"))
component = tables.component_scores(blocks, y)

## tab:errcorr

Each component method on its own, its gap to the volumetric baseline, and how far its per-case
errors depart from the volumetric ones. The correlation is measured **within each grade**, so
agreement driven by the grade itself does not inflate it; a low value means the two methods fail
on different cases.

Expected:

| Method | AUC alone | $\Delta$ vs. Vol. | Err. corr. vs. Vol. |
|---|---|---|---|
| Volumetric features | 0.925 | --- | --- |
| MedGemma | 0.786 | -0.139 | 0.298 |
| MedGemma (few-shot) | 0.804 | -0.121 | **0.245** |
| BiomedCLIP (T1ce) | 0.797 | -0.128 | 0.298 |
| BiomedCLIP (enhancement) | 0.855 | -0.070 | 0.483 |

No embedding matches the volumetric baseline alone, and the MedGemma few-shot errors are the
least correlated with it. It is not the strongest embedding on its own (that is BiomedCLIP on the
enhancement map), which is the ordering the fusion result below follows.

In [ ]:
errcorr = tables.table_errcorr(blocks, y, component)
display(errcorr)

published = {
    "Volumetric features": ("0.925", "---", "---"),
    "MedGemma": ("0.786", "-0.139", "0.298"),
    "MedGemma (few-shot)": ("0.804", "-0.121", "0.245"),
    "BiomedCLIP (T1ce)": ("0.797", "-0.128", "0.298"),
    "BiomedCLIP (enhancement)": ("0.855", "-0.070", "0.483"),
}
check = tables.check_printed(errcorr, {
    row: dict(zip(["AUC alone", "$\\Delta$ vs. Vol.", "Err. corr. vs. Vol."], values))
    for row, values in published.items()})
display(check)
print("all cells match the paper:", bool(check["match"].all()))

### The full pairwise picture

Three ways of asking the same question. The paper quotes the middle one, and also quotes from it
that the MedGemma few-shot and BiomedCLIP enhancement errors correlate 0.432 with each other.

In [ ]:
matrices = tables.errcorr_matrices(blocks, y, component)
for name, matrix in matrices.items():
    print(f"--- {name} ---")
    display(matrix)

pair = matrices["within grade"].loc["MedGemma (few-shot)", "BiomedCLIP (enhancement)"]
print(f"few-shot against BiomedCLIP enhancement, within grade: {pair:.3f} "
      f"(paper 0.432, match {f'{pair:.3f}' == '0.432'})")

## tab:matrix

Grading AUC as the components of each embedding are appended to the seventeen volumetric features.
The AUC is pooled out-of-fold with a random forest, and each difference is resampled against the
volumetric baseline over cases, 2000 times. $\Delta$ is the printed AUC minus the printed baseline
of 0.925, and each interval is rounded from its four-decimal form.

Expected:

| Feature set | AUC | $\Delta$ | 95% CI |
|---|---|---|---|
| Volumetric features | 0.925 | --- | --- |
| + MedGemma | 0.928 | +0.003 | [-0.008, +0.019] |
| + MedGemma (few-shot) | **0.937** | +0.012 | [+0.001, +0.025] |
| + BiomedCLIP (T1ce) | 0.926 | +0.001 | [-0.008, +0.009] |
| + BiomedCLIP (enhancement) | 0.927 | +0.002 | [-0.008, +0.012] |
| + MedGemma + BiomedCLIP (T1ce) | 0.922 | -0.003 | [-0.017, +0.015] |
| + MedGemma (few-shot) + BiomedCLIP (enhancement) | 0.934 | +0.009 | [-0.004, +0.022] |

The few-shot embedding is the only row whose interval lies above zero, from 0.925 to 0.937, and
its lower end sits at +0.001. BiomedCLIP gains at most +0.002, and keeping its components
alongside the few-shot ones lowers the result slightly, to 0.934, rather than adding to it.

In [ ]:
matrix = tables.table_matrix(blocks, y)
display(matrix)

published = {
    "Volumetric features": ("0.925", "---", "---"),
    "$+$ MedGemma": ("0.928", "+0.003", "[-0.008, +0.019]"),
    "$+$ MedGemma (few-shot)": ("0.937", "+0.012", "[+0.001, +0.025]"),
    "$+$ BiomedCLIP (T1ce)": ("0.926", "+0.001", "[-0.008, +0.009]"),
    "$+$ BiomedCLIP (enhancement)": ("0.927", "+0.002", "[-0.008, +0.012]"),
    "$+$ MedGemma $+$ BiomedCLIP (T1ce)": ("0.922", "-0.003", "[-0.017, +0.015]"),
    "$+$ MedGemma (few-shot) $+$ BiomedCLIP (enhancement)": ("0.934", "+0.009", "[-0.004, +0.022]"),
}
check = tables.check_printed(matrix, {
    row: dict(zip(["AUC", "$\\Delta$ vs. Volumetric", "95\\% CI"], values))
    for row, values in published.items()})
display(check)
print("all cells match the paper:", bool(check["match"].all()))

In [ ]:
detail = tables.table_matrix(blocks, y, detail=True)
display(detail)
p_fewshot = detail.loc["$+$ MedGemma (few-shot)", "P($\\Delta > 0$)"]
print(f"P(gain > 0) for the few-shot row: {p_fewshot:.4f}, "
      f"printed in the text as {stats.round_half_up(p_fewshot, 2):.2f} (paper 0.98)")

At four decimals the few-shot difference is +0.0123, with a 0.98 probability of being positive
and an interval whose lower end is +0.0009. That is one assignment of cases to folds. The next
section repeats it over five.

## Repeated cross-validation

The single run above uses one assignment of cases to folds, seed 42. Re-running the fusion rows
the paper discusses over five assignments (seeds 42, 1, 2, 3 and 4; the forest, PCA and bootstrap
seeds unchanged) shows how much of the seed-42 gain belongs to that split. Every difference is
paired with the volumetric baseline of the same seed.

Expected mean gain: +0.0052 (positive on 5 of 5 seeds) for the baseline prompt, **+0.0035**
(4 of 5) for the few-shot prompt, +0.0049 (5 of 5) for the three-way row. Seed 42 is the most
favourable of the five for the few-shot row, and over the five seeds the few-shot prompt does not
add more than the plain prompt does. The paper pairs every seed-42 fusion number with this mean.

In [ ]:
repeated = tables.table_repeated_cv(blocks, y)
display(repeated.T)

expected = {"$+$ MedGemma": ("+0.0052", "5/5"),
            "$+$ MedGemma (few-shot)": ("+0.0035", "4/5"),
            "$+$ MedGemma (few-shot) $+$ BiomedCLIP (enhancement)": ("+0.0049", "5/5")}
check = tables.check_printed(repeated, {
    row: {"mean $\\Delta$": mean, "seeds with $\\Delta > 0$": seeds}
    for row, (mean, seeds) in expected.items()})
display(check)
print("all cells match:", bool(check["match"].all()))

## The routing alternative

Concatenation lets the classifier weigh the two views globally. Routing would decide per case. The
gate is MedGemma's own confidence in the grade it wrote: the saved answer is teacher-forced up to
the assessment field and the next-token distribution is read, so this is the probability the model
actually decided with rather than a classifier fitted on top of its hidden state.

In [ ]:
p_native = routing.load_native_confidence(config.MEDGEMMA_NATIVE_CONF, cases)
p_volumetric = component["Volumetric features"]
p_medgemma = component["MedGemma"]
p_clip = component["BiomedCLIP (T1ce)"]
threshold_volumetric = stats.youden_threshold(y, p_volumetric)
threshold_medgemma = stats.youden_threshold(y, p_medgemma)
threshold_clip = stats.youden_threshold(y, p_clip)

standalone = routing.standalone_table(y, {
    "MedGemma, its own grade token": (p_native, 0.5),
    "MedGemma, a forest over its hidden state": (p_medgemma, threshold_medgemma),
    "Volumetric features": (p_volumetric, threshold_volumetric),
})
display(standalone)
expected = {"MedGemma, its own grade token": (0.518, 289),
            "MedGemma, a forest over its hidden state": (0.786, 300),
            "Volumetric features": (0.925, 334)}
print("matches the expected values:", all(
    (standalone.loc[k, "AUC"], standalone.loc[k, "Correct of 369"]) == v
    for k, v in expected.items()))

In [ ]:
saturated = routing.saturation(p_native)
display(saturated)
print("cases within 0.01 of a hard 0 or 1:", int(saturated["value"].iloc[1]), "(expected 324)")

MedGemma's own grade is close to chance and its confidence is heavily peaked. A saturated signal
carries little information to gate on, which is the first hint that no threshold on it will work.

### The router itself

A growing share of cases is handed to the embedding, most-confident-MedGemma first. The baseline
sits at 0% deferral, so any row that helps has to beat it.

In [ ]:
sweep, blended = routing.deferral_sweep(
    y, p_volumetric, threshold_volumetric, p_medgemma, threshold_medgemma,
    gate=routing.confidence(p_native))
display(sweep)
print(f"volumetric baseline AUC: {sweep.loc['0%', 'AUC']}")
print("AUC at 5% ... 50% deferred:", sweep["AUC"].tolist()[1:],
      "(expected [0.914, 0.909, 0.896, 0.89, 0.884, 0.865])")

In [ ]:
best = routing.best_router_bootstrap(
    y, p_volumetric, sweep, blended, "native-confidence gate to the embedding")
display(best)
print("expected: 5%, -0.0111, [-0.0270, -0.0000], 0.025 | match:",
      best.iloc[0].tolist() == ["5%", "-0.0111", "[-0.0270, -0.0000]", 0.025])

Bootstrapping the best point of the sweep is deliberately generous: the fraction was picked after
seeing the results, so this is the most favourable test available. It still does not win.

### The comparative router

Instead of gating on one confidence, defer the cases where MedGemma is the more confident of the
two. Both confidences are turned into percentiles first, since a token probability and a forest
probability live on different scales. Expected: best fraction 10%, $\Delta$ AUC -0.0041,
interval [-0.0163, +0.0066].

In [ ]:
comparative = (routing.percentile(routing.confidence(p_native))
               - routing.percentile(routing.confidence(p_volumetric, threshold_volumetric)))
sweep_c, blended_c = routing.deferral_sweep(
    y, p_volumetric, threshold_volumetric, p_medgemma, threshold_medgemma, gate=comparative)
display(sweep_c)
best_c = routing.best_router_bootstrap(
    y, p_volumetric, sweep_c, blended_c, "comparative-confidence router")
display(best_c)
print("expected: 10%, -0.0041, [-0.0163, +0.0066], 0.269 | match:",
      best_c.iloc[0].tolist() == ["10%", "-0.0041", "[-0.0163, +0.0066]", 0.269])

### Gates built from the two views' own scores

The same sweep with three label-free gates that need no MedGemma token at all: hand over the cases
the volumetric classifier is least sure of, the cases where the two views disagree most, or the
cases the volumetric classifier calls most low-grade. Each is tried with the MedGemma and with the
BiomedCLIP embedding. No row's best fraction improves on the volumetric baseline; the best AUC of
every sweep is the 0% row itself.

In [ ]:
gates = {
    "least volumetric confidence": (routing.confidence(p_volumetric, threshold_volumetric), False),
    "largest disagreement": (None, True),
    "most low-grade volumetric score": (p_volumetric, True),
}
rows = []
for name, p_other, threshold_other in [("MedGemma", p_medgemma, threshold_medgemma),
                                       ("BiomedCLIP (T1ce)", p_clip, threshold_clip)]:
    for rule, (gate, most_first) in gates.items():
        gate = np.abs(p_volumetric - p_other) if gate is None else gate
        sweep_g, blended_g = routing.deferral_sweep(
            y, p_volumetric, threshold_volumetric, p_other, threshold_other,
            gate=gate, most_confident_first=most_first)
        best_g = routing.best_router_bootstrap(y, p_volumetric, sweep_g, blended_g, rule)
        rows.append({"embedding": name, "gate": rule,
                     "best AUC at any fraction": sweep_g["AUC"].max(),
                     "at": sweep_g["AUC"].idxmax(),
                     **best_g.iloc[0].to_dict()})
gated = pd.DataFrame(rows).set_index(["embedding", "gate"])
display(gated)
expected = ["-0.0068", "-0.0788", "-0.0289", "-0.0109", "-0.0559", "-0.0300"]
print("best-net Delta AUC per router:", gated["$\\Delta$ AUC"].tolist())
print("matches the expected values:", gated["$\\Delta$ AUC"].tolist() == expected,
      "| every best AUC is the 0% baseline:", bool((gated["at"] == "0%").all()))

### Why nothing routes

The decisive test. Restrict attention to the cases where the two views disagree, since those are
the only ones a router can change, and ask whether any label-free signal separates the cases where
deferring **rescues** the volumetric classifier from those where it **breaks** it. A signal that
could drive a router scores far from 0.5.

In [ ]:
flip, summary = routing.flip_separation(
    y, p_volumetric, threshold_volumetric, p_medgemma, threshold_medgemma,
    signals={
        "embedding confidence": routing.confidence(p_medgemma, threshold_medgemma),
        "disagreement between the two views": np.abs(p_volumetric - p_medgemma),
        "MedGemma's own confidence": routing.confidence(p_native),
        "MedGemma's own P(low-grade)": p_native,
        "volumetric confidence": routing.confidence(p_volumetric, threshold_volumetric),
        "comparative confidence": comparative,
        "volumetric P(low-grade), circular": p_volumetric,
    })
print(summary)
display(flip)
expected = [0.682, 0.241, 0.413, 0.588, 0.303, 0.606, 0.605]
print("matches the expected values:", flip.iloc[:, 0].tolist() == expected,
      "| disagreements 74, rescuable 20, breakable 54:",
      summary == {"disagreements": 74, "rescuable": 20, "breakable": 54})

MedGemma's own signals sit near chance: its confidence separates a rescue from a break at 0.413
and its own P(low-grade) at 0.588. Two signals sit further from 0.5 on these 74 disagreements,
the embedding's confidence (0.682) and the disagreement between the two views (0.241), and the
circular one is unusable by construction: rescuable and breakable are *defined* by whether the
volumetric classifier is wrong, which its own probability sets. None of this turns into a router
that works. Every gate tried above, the native confidence, the comparative one and the three built
from the two views' own scores, gives an AUC below the volumetric baseline at its best fraction.

So the conclusion the paper draws is the sharper one. It is not that the embeddings repeat the
volumetric mistakes: `tab:errcorr` shows they do not, and MedGemma and the volumetric features
genuinely fail on different cases. It is that the complementarity has not been turned into a
routing rule that beats the volumetric baseline. Concatenation is therefore the fusion reported.